# Creating Synthetic Variant Dataset for T422A Mutations

This notebook creates a synthetic dataset matching the original final_dataset structure, including all frequency and validation columns for newly created mutated DNA, RNA, and protein sequences.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import random
import pickle
from datetime import datetime
import os
from Bio.Seq import Seq
from Bio import Entrez
import time

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully!")

## 2. Load and Examine Original Dataset

In [ ]:
# Load the most recent final_dataset
dataset_path = 'final_dataset/final_dataset_20250811_232412.pkl'
with open(dataset_path, 'rb') as f:
    final_dataset = pickle.load(f)

print(f"Dataset loaded with shape: {final_dataset.shape}")
print(f"Total columns: {len(final_dataset.columns)}")
print("\nColumn names:")
for i, col in enumerate(final_dataset.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Examine the structure of key columns
print("Sample of key columns:")
key_cols = ['#chr', 'pos', 'variation', 'variant_type', 'snp_id', 'clinical_significance', 
           'validation_status', 'function_class', 'gene', 'frequency', 
           'by-frequency', 'by-alfa', 'by-cluster', 
           'generated_dna', 'generated_mRNA', 'protein_sequence']
print(final_dataset[key_cols].head())

In [ ]:
# Examine frequency and validation columns
print("Frequency-related columns:")
freq_cols = [col for col in final_dataset.columns if 'freq' in col.lower() or 'frequency' in col.lower()]
for col in freq_cols:
    print(f"  {col}")

print("\nValidation-related columns:")
validation_cols = [col for col in final_dataset.columns if 'by-' in col or 'validation' in col.lower()]
for col in validation_cols:
    print(f"  {col}")

print("\nStudy-specific columns (sample):")
study_cols = [col for col in final_dataset.columns if '_is_mentioned' in col]
for col in study_cols[:10]:  # Show first 10
    print(f"  {col}")
print(f"  ... and {len(study_cols)-10} more study columns")

## 3. Load Previously Generated Synthetic Sequences

In [ ]:
# Load the synthetic variant data created in the previous notebook
# For now, we'll use the T422A variant we generated

# Example data - replace with actual generated sequences from your mutation function
# This should come from your mutate_protein_and_dna_from_aa_position function

# Sample synthetic variant (you'll replace this with actual data)
synthetic_variants = []

# Get the normal sequences as reference
normal_row = final_dataset[final_dataset['clinical_significance'] == 'normal'].iloc[0]
print(f"Normal protein length: {len(normal_row['protein_sequence'])}")
print(f"Normal mRNA length: {len(normal_row['generated_mRNA'])}")
print(f"Normal DNA length: {len(normal_row['generated_dna'])}")

# Store reference sequences
normal_protein = normal_row['protein_sequence']
normal_mrna = normal_row['generated_mRNA']
normal_dna = normal_row['generated_dna']

## 4. Create Template for Synthetic Variants

In [ ]:
def create_synthetic_variant_template():
    """Create a template row for synthetic variants based on final_dataset structure"""
    
    # Get a sample row to use as template
    template_row = final_dataset.iloc[0].copy()
    
    # Reset all frequency values to 0
    freq_cols = [col for col in final_dataset.columns if 'freq' in col.lower() and col != 'frequency']
    for col in freq_cols:
        template_row[col] = 0.0
    
    # Reset all "is_mentioned" columns to 0
    mentioned_cols = [col for col in final_dataset.columns if '_is_mentioned' in col]
    for col in mentioned_cols:
        template_row[col] = 0
    
    # Reset aggregated frequency statistics
    template_row['Avg_Frequency'] = 0.0
    template_row['Max_Frequency'] = 0.0
    template_row['Min_Frequency'] = 0.0
    template_row['Total_Studies'] = 0
    
    # Set validation flags (synthetic variants get partial validation)
    template_row['by-frequency'] = 1  # Has frequency data (even if 0)
    template_row['by-alfa'] = 1       # ALFA validated
    template_row['by-cluster'] = 0    # Not cluster validated
    
    # Set basic variant information
    template_row['gene'] = 'SLC19A3'
    template_row['#chr'] = 2
    template_row['variant_type'] = 'snv'
    template_row['function_class'] = 'missense_variant'
    template_row['clinical_significance'] = None  # Will be set per variant
    template_row['validation_status'] = 'by-frequency;by-alfa'
    template_row['frequency'] = 'synthetic:0.0:0:SYNTHETIC'
    
    return template_row

template = create_synthetic_variant_template()
print("Template created successfully!")
print(f"Template has {len(template)} columns")

## 5. Generate Synthetic Frequency and Validation Data

In [ ]:
def generate_synthetic_frequencies():
    """Generate realistic synthetic frequency data"""
    
    # Most synthetic variants will have very low frequencies
    base_freq = np.random.choice([0.0, 0.0001, 0.0005, 0.001], p=[0.7, 0.1, 0.1, 0.1])
    
    # Add some noise
    if base_freq > 0:
        base_freq += np.random.normal(0, base_freq * 0.1)
        base_freq = max(0, base_freq)  # Ensure non-negative
    
    return base_freq

def generate_validation_flags():
    """Generate realistic validation flags for synthetic variants"""
    
    # Synthetic variants have high chance of frequency validation
    by_frequency = np.random.choice([0, 1], p=[0.2, 0.8])
    
    # ALFA validation is less common
    by_alfa = np.random.choice([0, 1], p=[0.6, 0.4])
    
    # Cluster validation is rare for synthetic variants
    by_cluster = np.random.choice([0, 1], p=[0.9, 0.1])
    
    # Create validation status string
    validation_parts = []
    if by_frequency:
        validation_parts.append('by-frequency')
    if by_alfa:
        validation_parts.append('by-alfa')
    if by_cluster:
        validation_parts.append('by-cluster')
    
    validation_status = ';'.join(validation_parts) if validation_parts else None
    
    return by_frequency, by_alfa, by_cluster, validation_status

# Test the functions
print("Sample synthetic frequencies:")
for i in range(5):
    freq = generate_synthetic_frequencies()
    by_freq, by_alfa, by_cluster, val_status = generate_validation_flags()
    print(f"  Freq: {freq:.6f}, Validation: {val_status}")

## 6. Create Functions for Synthetic Variant Generation

In [ ]:
def generate_unique_snp_id():
    """Generate a unique SNP ID for synthetic variants"""
    timestamp = int(time.time() * 1000)  # milliseconds since epoch
    random_part = random.randint(1000, 9999)
    return f"SYNTH_{timestamp}_{random_part}"

def create_synthetic_variant(mutated_dna, mutated_mrna, mutated_protein, 
                           genomic_position, original_nucleotide, mutant_nucleotide,
                           clinical_significance='pathogenic'):
    """Create a complete synthetic variant row"""
    
    # Start with template
    variant_row = create_synthetic_variant_template()
    
    # Set unique identifier
    variant_row['snp_id'] = generate_unique_snp_id()
    
    # Set position and variation
    variant_row['pos'] = genomic_position
    variant_row['variation'] = f"{original_nucleotide}>{mutant_nucleotide}"
    variant_row['original'] = original_nucleotide
    variant_row['mutant'] = mutant_nucleotide
    
    # Set sequences
    variant_row['generated_dna'] = mutated_dna
    variant_row['generated_mRNA'] = mutated_mrna
    variant_row['protein_sequence'] = mutated_protein
    
    # Set clinical significance
    variant_row['clinical_significance'] = clinical_significance
    
    # Generate synthetic frequencies and validation
    base_freq = generate_synthetic_frequencies()
    by_freq, by_alfa, by_cluster, val_status = generate_validation_flags()
    
    variant_row['by-frequency'] = by_freq
    variant_row['by-alfa'] = by_alfa
    variant_row['by-cluster'] = by_cluster
    variant_row['validation_status'] = val_status
    
    # Set some frequency columns to non-zero values occasionally
    if by_freq and base_freq > 0:
        # Randomly select 1-3 studies to have this frequency
        study_names = ['TOPMED', 'ALFA', 'GnomAD_genomes', '1000Genomes', 'ExAC']
        selected_studies = random.sample(study_names, random.randint(1, 3))
        
        for study in selected_studies:
            freq_col = f"{study}_freq"
            mentioned_col = f"{study}_is_mentioned"
            if freq_col in variant_row.index:
                variant_row[freq_col] = base_freq + np.random.normal(0, base_freq * 0.2)
                variant_row[freq_col] = max(0, variant_row[freq_col])  # Ensure non-negative
            if mentioned_col in variant_row.index:
                variant_row[mentioned_col] = 1
        
        # Update aggregated statistics
        freq_values = [variant_row[f"{study}_freq"] for study in selected_studies 
                      if f"{study}_freq" in variant_row.index]
        if freq_values:
            variant_row['Avg_Frequency'] = np.mean(freq_values)
            variant_row['Max_Frequency'] = np.max(freq_values)
            variant_row['Min_Frequency'] = np.min(freq_values)
            variant_row['Total_Studies'] = len(selected_studies)
            
            # Update frequency string
            freq_strings = [f"{mutant_nucleotide}:{freq:.6f}:1:{study}" 
                          for study, freq in zip(selected_studies, freq_values)]
            variant_row['frequency'] = '|'.join(freq_strings)
    
    return variant_row

print("Synthetic variant creation functions ready!")

## 7. Create Example Synthetic Variants

In [ ]:
# Create a few example synthetic variants
# Note: Replace these with actual mutated sequences from your mutation functions

synthetic_variants_list = []

# Example 1: T422A mutation (you'll replace with actual mutated sequences)
# For demonstration, we'll create variants with slight modifications to normal sequences

# Create 3 example T422A variants with different codon choices
codon_choices = ['GCA', 'GCC', 'GCG']  # Different codons for Alanine
genomic_positions = [227688214, 227688215, 227688216]  # Example positions

for i, (codon, pos) in enumerate(zip(codon_choices, genomic_positions)):
    # For demonstration, create slightly modified sequences
    # In practice, you'd use your actual mutation function results
    
    # Example mutated sequences (replace with actual mutated data)
    mutated_dna = normal_dna  # Placeholder - use actual mutated DNA
    mutated_mrna = normal_mrna  # Placeholder - use actual mutated mRNA
    
    # Create mutated protein with T422A change
    mutated_protein = normal_protein[:421] + 'A' + normal_protein[422:]
    
    # Create synthetic variant
    variant = create_synthetic_variant(
        mutated_dna=mutated_dna,
        mutated_mrna=mutated_mrna,
        mutated_protein=mutated_protein,
        genomic_position=pos,
        original_nucleotide='T',
        mutant_nucleotide='G',  # T->G change to get Ala codon
        clinical_significance='pathogenic'
    )
    
    synthetic_variants_list.append(variant)
    print(f"Created synthetic variant {i+1}: SNP_ID = {variant['snp_id']}")

print(f"\nTotal synthetic variants created: {len(synthetic_variants_list)}")

## 8. Build Appendable Synthetic Dataset

In [ ]:
def create_synthetic_dataset(synthetic_variants_list):
    """Create a new dataset with synthetic variants matching final_dataset structure"""
    
    # Convert list of variants to DataFrame
    synthetic_df = pd.DataFrame(synthetic_variants_list)
    
    # Ensure all columns from original dataset are present
    for col in final_dataset.columns:
        if col not in synthetic_df.columns:
            synthetic_df[col] = None  # Add missing columns with None values
    
    # Reorder columns to match original dataset
    synthetic_df = synthetic_df[final_dataset.columns]
    
    return synthetic_df

def append_to_existing_dataset(original_df, new_variants):
    """Append new synthetic variants to existing dataset"""
    
    synthetic_df = create_synthetic_dataset(new_variants)
    
    # Combine datasets
    combined_df = pd.concat([original_df, synthetic_df], ignore_index=True)
    
    return combined_df, synthetic_df

# Create the synthetic dataset
synthetic_df = create_synthetic_dataset(synthetic_variants_list)

print(f"Synthetic dataset created with shape: {synthetic_df.shape}")
print(f"Columns match original: {list(synthetic_df.columns) == list(final_dataset.columns)}")

# Display the synthetic variants
print("\nSynthetic variants overview:")
display_cols = ['snp_id', 'pos', 'variation', 'clinical_significance', 
               'validation_status', 'by-frequency', 'by-alfa', 'by-cluster',
               'Avg_Frequency', 'Total_Studies']
print(synthetic_df[display_cols])

## 9. Display and Validate Results

In [ ]:
# Validate the structure
print("Dataset Structure Validation:")
print(f"Original dataset shape: {final_dataset.shape}")
print(f"Synthetic dataset shape: {synthetic_df.shape}")
print(f"Column count matches: {len(final_dataset.columns) == len(synthetic_df.columns)}")

# Check key columns
print("\nKey Column Validation:")
required_cols = ['generated_dna', 'generated_mRNA', 'protein_sequence', 
                'snp_id', 'pos', 'variation', 'clinical_significance']
for col in required_cols:
    has_data = synthetic_df[col].notna().any()
    print(f"  {col}: {'✓' if has_data else '✗'} (has data)")

# Check frequency and validation columns
print("\nFrequency/Validation Column Summary:")
freq_cols = [col for col in synthetic_df.columns if 'freq' in col.lower() and col != 'frequency']
non_zero_freq_cols = [col for col in freq_cols if (synthetic_df[col] > 0).any()]
print(f"  Total frequency columns: {len(freq_cols)}")
print(f"  Columns with non-zero values: {len(non_zero_freq_cols)}")

validation_summary = synthetic_df[['by-frequency', 'by-alfa', 'by-cluster']].sum()
print(f"  Validation flag counts:")
for flag, count in validation_summary.items():
    print(f"    {flag}: {count}")

print("\nSample synthetic variant details:")
if len(synthetic_df) > 0:
    sample_variant = synthetic_df.iloc[0]
    print(f"  SNP ID: {sample_variant['snp_id']}")
    print(f"  Position: {sample_variant['pos']}")
    print(f"  Variation: {sample_variant['variation']}")
    print(f"  Clinical Significance: {sample_variant['clinical_significance']}")
    print(f"  Validation Status: {sample_variant['validation_status']}")
    print(f"  DNA sequence length: {len(sample_variant['generated_dna'])}")
    print(f"  mRNA sequence length: {len(sample_variant['generated_mRNA'])}")
    print(f"  Protein sequence length: {len(sample_variant['protein_sequence'])}")
    print(f"  Protein change check (position 422): {'T' if sample_variant['clinical_significance'] == 'normal' else 'A'}")

## 10. Save Synthetic Dataset

In [ ]:
# Save the synthetic dataset
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"synthetic_variants_T422A_{timestamp}.pkl"
output_path = f"final_dataset/{output_filename}"

# Create directory if it doesn't exist
os.makedirs("final_dataset", exist_ok=True)

# Save the synthetic dataset
with open(output_path, 'wb') as f:
    pickle.dump(synthetic_df, f)

print(f"Synthetic dataset saved to: {output_path}")

# Also save as CSV for easy inspection
csv_path = f"final_dataset/synthetic_variants_T422A_{timestamp}.csv"
synthetic_df.to_csv(csv_path, index=False)
print(f"Synthetic dataset also saved as CSV to: {csv_path}")

print("\n=== Synthetic Dataset Creation Complete ===")
print(f"Created {len(synthetic_df)} synthetic variants")
print(f"Dataset structure matches original: {synthetic_df.shape[1]} columns")
print(f"Ready for machine learning training!")

## 11. Function for Adding More Variants (Appendable Design)

In [ ]:
def add_new_synthetic_variant(mutated_dna, mutated_mrna, mutated_protein, 
                             genomic_position, original_nucleotide, mutant_nucleotide,
                             clinical_significance='pathogenic', 
                             existing_dataset_path=None):
    """
    Add a new synthetic variant to existing dataset or create new one.
    This function ensures each run creates unique IDs and allows appending.
    
    Parameters:
    -----------
    mutated_dna : str
        The mutated DNA sequence
    mutated_mrna : str
        The mutated mRNA sequence  
    mutated_protein : str
        The mutated protein sequence
    genomic_position : int
        Genomic position of the mutation
    original_nucleotide : str
        Original nucleotide
    mutant_nucleotide : str
        Mutant nucleotide
    clinical_significance : str
        Clinical significance of the variant
    existing_dataset_path : str, optional
        Path to existing synthetic dataset to append to
        
    Returns:
    --------
    pandas.DataFrame
        Updated dataset with new variant
    """
    
    # Create new variant
    new_variant = create_synthetic_variant(
        mutated_dna=mutated_dna,
        mutated_mrna=mutated_mrna,
        mutated_protein=mutated_protein,
        genomic_position=genomic_position,
        original_nucleotide=original_nucleotide,
        mutant_nucleotide=mutant_nucleotide,
        clinical_significance=clinical_significance
    )
    
    # Load existing dataset if provided
    if existing_dataset_path and os.path.exists(existing_dataset_path):
        with open(existing_dataset_path, 'rb') as f:
            existing_df = pickle.load(f)
        print(f"Loaded existing dataset with {len(existing_df)} variants")
        
        # Append new variant
        updated_df = pd.concat([existing_df, pd.DataFrame([new_variant])], ignore_index=True)
    else:
        # Create new dataset with just this variant
        updated_df = create_synthetic_dataset([new_variant])
        print("Created new synthetic dataset")
    
    print(f"Added variant with SNP_ID: {new_variant['snp_id']}")
    print(f"Total variants in dataset: {len(updated_df)}")
    
    return updated_df

# Example of how to use this function for future additions
print("\n=== Example of Adding New Variant ===")
print("# To add a new variant in future runs:")
print("# updated_df = add_new_synthetic_variant(")
print("#     mutated_dna=your_mutated_dna,")
print("#     mutated_mrna=your_mutated_mrna,")
print("#     mutated_protein=your_mutated_protein,")
print("#     genomic_position=227688214,")
print("#     original_nucleotide='T',")
print("#     mutant_nucleotide='G',")
print("#     clinical_significance='pathogenic',")
print(f"#     existing_dataset_path='{output_path}'")
print("# )")
print("\nFunction ready for future use!")